In [ ]:
# =============================================================================
# SPAM DETECTION USING MACHINE LEARNING
# =============================================================================
# Author      : [Your Name]
# Institution : MSc Data Science
# Description : End-to-end spam detection pipeline with EDA, feature
#               engineering, and comparison of multiple ML classifiers.
# Dataset     : SMS Spam Collection (or any CSV with 'label' and 'text' cols)
# =============================================================================

# ── Imports ───────────────────────────────────────────────────────────────────
import os
import re
import string
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from collections import Counter
from wordcloud import WordCloud

# Scikit-learn
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_curve, auc,
    ConfusionMatrixDisplay
)

# Models
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# NLP
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

warnings.filterwarnings("ignore")
nltk.download("stopwords", quiet=True)
nltk.download("punkt", quiet=True)

# ── Matplotlib style ──────────────────────────────────────────────────────────
plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.family": "sans-serif",
})
PALETTE = {"ham": "#4CAF50", "spam": "#E53935"}

# =============================================================================
# 1. LOAD DATASET
# =============================================================================

def load_dataset(path: str) -> pd.DataFrame:
    """
    Load the spam dataset from a CSV file.

    Expected columns (flexible):
        • Two-column TSV/CSV  →  label | text         (SMS Spam Collection format)
        • Generic CSV         →  any column named 'label'/'class' + 'text'/'message'/'sms'

    Parameters
    ----------
    path : str
        Absolute or relative path to the dataset file.

    Returns
    -------
    pd.DataFrame with columns ['label', 'text'].

    Usage
    -----
    >>> df = load_dataset("data/spam.csv")          # generic CSV
    >>> df = load_dataset("data/SMSSpamCollection") # tab-separated, no header

    Download the SMS Spam Collection:
        https://archive.ics.uci.edu/ml/datasets/SMS+Spam+Collection
    """
    ext = os.path.splitext(path)[-1].lower()

    # ── Tab-separated, no header (original SMS Spam Collection) ──────────────
    if ext in ("", ".tsv", ".txt"):
        df = pd.read_csv(path, sep="\t", header=None, names=["label", "text"],
                         encoding="latin-1")

    # ── CSV ───────────────────────────────────────────────────────────────────
    elif ext == ".csv":
        df = pd.read_csv(path, encoding="latin-1")
        df = df.dropna(axis=1, how="all")           # drop fully-empty columns

        # Normalise column names to lower-case, strip whitespace
        df.columns = df.columns.str.strip().str.lower()

        # Find the label column
        label_candidates = ["label", "class", "category", "spam", "v1"]
        label_col = next((c for c in label_candidates if c in df.columns), None)
        if label_col is None:
            raise ValueError(
                f"Cannot find a label column. Columns found: {list(df.columns)}\n"
                "Please rename your label column to 'label'."
            )

        # Find the text column
        text_candidates = ["text", "message", "sms", "email", "body", "v2", "content"]
        text_col = next((c for c in text_candidates if c in df.columns), None)
        if text_col is None:
            raise ValueError(
                f"Cannot find a text column. Columns found: {list(df.columns)}\n"
                "Please rename your text column to 'text'."
            )

        df = df.rename(columns={label_col: "label", text_col: "text"})[["label", "text"]]

    else:
        raise ValueError(f"Unsupported file extension: '{ext}'. Use .csv or .tsv/.txt")

    # ── Sanitise labels ───────────────────────────────────────────────────────
    df["label"] = df["label"].astype(str).str.strip().str.lower()
    df = df[df["label"].isin(["ham", "spam"])].reset_index(drop=True)

    print(f"✔  Dataset loaded   → {df.shape[0]:,} rows  |  "
          f"spam={df['label'].value_counts().get('spam', 0):,}  "
          f"ham={df['label'].value_counts().get('ham', 0):,}")
    return df


# =============================================================================
# 2. EXPLORATORY DATA ANALYSIS  (EDA)
# =============================================================================

def run_eda(df: pd.DataFrame, save_plots: bool = False, plot_dir: str = "plots") -> None:
    """
    Comprehensive EDA:
        1. Class distribution
        2. Message length distributions (chars & words)
        3. Top-N frequent words per class
        4. Word clouds
        5. Correlation heatmap of numeric features
    """
    os.makedirs(plot_dir, exist_ok=True)

    print("\n" + "=" * 60)
    print("  EXPLORATORY DATA ANALYSIS")
    print("=" * 60)

    # ── Basic statistics ──────────────────────────────────────────────────────
    print("\n[1] Dataset shape   :", df.shape)
    print("[2] Missing values  :\n", df.isnull().sum())
    print("\n[3] Class distribution:\n", df["label"].value_counts())
    print(f"\n[4] Class balance  : "
          f"{df['label'].value_counts(normalize=True).mul(100).round(2).to_dict()}")

    # ── Derived features ──────────────────────────────────────────────────────
    df["char_count"]  = df["text"].str.len()
    df["word_count"]  = df["text"].str.split().str.len()
    df["sent_count"]  = df["text"].str.split(r"[.!?]").str.len()
    df["capital_ratio"] = df["text"].apply(
        lambda x: sum(1 for c in x if c.isupper()) / max(len(x), 1)
    )

    print("\n[5] Descriptive stats (numeric features):")
    print(df.groupby("label")[["char_count", "word_count", "capital_ratio"]].describe())

    # ── Figure 1 : Class distribution ─────────────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle("Class Distribution", fontsize=14, fontweight="bold")

    counts = df["label"].value_counts()
    axes[0].bar(counts.index, counts.values,
                color=[PALETTE[l] for l in counts.index], width=0.5, edgecolor="white")
    axes[0].set_title("Absolute Counts")
    axes[0].set_ylabel("Number of Messages")
    for i, (lbl, val) in enumerate(counts.items()):
        axes[0].text(i, val + 20, str(val), ha="center", fontweight="bold")

    axes[1].pie(counts.values, labels=counts.index,
                colors=[PALETTE[l] for l in counts.index],
                autopct="%1.1f%%", startangle=90,
                wedgeprops={"edgecolor": "white", "linewidth": 2})
    axes[1].set_title("Proportion")

    plt.tight_layout()
    _save_or_show(fig, save_plots, os.path.join(plot_dir, "01_class_distribution.png"))

    # ── Figure 2 : Message length distributions ────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle("Message Length Distributions", fontsize=14, fontweight="bold")

    for lbl, grp in df.groupby("label"):
        axes[0].hist(grp["char_count"], bins=60, alpha=0.65,
                     color=PALETTE[lbl], label=lbl, edgecolor="none")
        axes[1].hist(grp["word_count"], bins=40, alpha=0.65,
                     color=PALETTE[lbl], label=lbl, edgecolor="none")

    axes[0].set_title("Character Count"); axes[0].set_xlabel("Characters")
    axes[1].set_title("Word Count");      axes[1].set_xlabel("Words")
    for ax in axes:
        ax.set_ylabel("Frequency"); ax.legend()

    plt.tight_layout()
    _save_or_show(fig, save_plots, os.path.join(plot_dir, "02_length_distribution.png"))

    # ── Figure 3 : Box plots ──────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    fig.suptitle("Feature Distribution by Class", fontsize=14, fontweight="bold")
    features = ["char_count", "word_count", "capital_ratio"]
    titles   = ["Character Count", "Word Count", "Capital Letter Ratio"]

    for ax, feat, title in zip(axes, features, titles):
        data_ham  = df[df["label"] == "ham"][feat]
        data_spam = df[df["label"] == "spam"][feat]
        bp = ax.boxplot([data_ham, data_spam], patch_artist=True,
                        medianprops={"color": "white", "linewidth": 2})
        for patch, color in zip(bp["boxes"], [PALETTE["ham"], PALETTE["spam"]]):
            patch.set_facecolor(color)
        ax.set_xticklabels(["Ham", "Spam"])
        ax.set_title(title)

    plt.tight_layout()
    _save_or_show(fig, save_plots, os.path.join(plot_dir, "03_boxplots.png"))

    # ── Figure 4 : Top-20 words ────────────────────────────────────────────────
    stop_words = set(stopwords.words("english"))
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle("Top 20 Words per Class (after stopword removal)",
                 fontsize=14, fontweight="bold")

    for ax, lbl in zip(axes, ["ham", "spam"]):
        words = " ".join(df[df["label"] == lbl]["text"]).lower()
        words = re.sub(r"[^a-z\s]", "", words).split()
        words = [w for w in words if w not in stop_words and len(w) > 2]
        freq  = Counter(words).most_common(20)
        terms, freqs = zip(*freq)
        ax.barh(terms[::-1], freqs[::-1], color=PALETTE[lbl], edgecolor="none")
        ax.set_title(f"Class: {lbl.upper()}", color=PALETTE[lbl])
        ax.set_xlabel("Frequency")

    plt.tight_layout()
    _save_or_show(fig, save_plots, os.path.join(plot_dir, "04_top_words.png"))

    # ── Figure 5 : Word clouds ────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle("Word Clouds", fontsize=14, fontweight="bold")

    for ax, lbl in zip(axes, ["ham", "spam"]):
        text = " ".join(df[df["label"] == lbl]["text"])
        wc   = WordCloud(width=700, height=400, background_color="white",
                         colormap="RdYlGn" if lbl == "ham" else "Reds",
                         stopwords=stop_words, max_words=150).generate(text)
        ax.imshow(wc, interpolation="bilinear")
        ax.axis("off")
        ax.set_title(f"{lbl.upper()} Messages", fontsize=13)

    plt.tight_layout()
    _save_or_show(fig, save_plots, os.path.join(plot_dir, "05_wordclouds.png"))

    # ── Figure 6 : Correlation heatmap ────────────────────────────────────────
    numeric_df = df[["char_count", "word_count", "sent_count", "capital_ratio",
                      "label"]].copy()
    numeric_df["label_enc"] = (numeric_df["label"] == "spam").astype(int)
    numeric_df.drop(columns=["label"], inplace=True)

    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(numeric_df.corr(), annot=True, fmt=".2f", cmap="coolwarm",
                linewidths=0.5, ax=ax)
    ax.set_title("Feature Correlation Heatmap", fontsize=13, fontweight="bold")
    plt.tight_layout()
    _save_or_show(fig, save_plots, os.path.join(plot_dir, "06_correlation.png"))

    print("\n✔  EDA complete.\n")


def _save_or_show(fig: plt.Figure, save: bool, path: str) -> None:
    if save:
        fig.savefig(path, bbox_inches="tight")
        print(f"   Figure saved → {path}")
    else:
        plt.show()
    plt.close(fig)


# =============================================================================
# 3. TEXT PREPROCESSING
# =============================================================================

stemmer    = PorterStemmer()
stop_words = set(stopwords.words("english"))


def preprocess_text(text: str) -> str:
    """
    Full NLP preprocessing pipeline:
        1. Lower-case
        2. Remove URLs, email addresses, phone numbers
        3. Remove punctuation & digits
        4. Tokenise
        5. Remove stop words
        6. Apply Porter stemming
    """
    text = text.lower()
    text = re.sub(r"http\S+|www\S+|https\S+", " ", text)           # URLs
    text = re.sub(r"\S+@\S+", " ", text)                            # emails
    text = re.sub(r"\b\d[\d\s\-().+]{7,}\b", " ", text)            # phone numbers
    text = re.sub(r"[^a-z\s]", " ", text)                           # punctuation/digits
    tokens = text.split()
    tokens = [stemmer.stem(t) for t in tokens
              if t not in stop_words and len(t) > 2]
    return " ".join(tokens)


def preprocess_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    """Apply preprocess_text to every row and return df with new column."""
    print("Preprocessing text …", end=" ", flush=True)
    df = df.copy()
    df["clean_text"] = df["text"].apply(preprocess_text)
    print("done.")
    return df


# =============================================================================
# 4. FEATURE ENGINEERING & TRAIN/TEST SPLIT
# =============================================================================

def prepare_features(df: pd.DataFrame, test_size: float = 0.20,
                     random_state: int = 42):
    """
    Encode labels and split data.

    Returns
    -------
    X_train, X_test, y_train, y_test
    """
    le = LabelEncoder()
    y  = le.fit_transform(df["label"])          # ham=0, spam=1
    X  = df["clean_text"]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state, stratify=y
    )
    print(f"✔  Train: {len(X_train):,}  |  Test: {len(X_test):,}  "
          f"(stratified, spam ratio ≈ {y_test.mean()*100:.1f}%)")
    return X_train, X_test, y_train, y_test, le


# =============================================================================
# 5. BUILD ML PIPELINES
# =============================================================================

def build_pipelines() -> dict:
    """
    Returns three scikit-learn pipelines:
        • Multinomial Naive Bayes     (fast, strong baseline for text)
        • Logistic Regression         (interpretable linear model)
        • Random Forest               (ensemble, handles non-linearity)

    Each pipeline = TF-IDF vectoriser  →  Classifier
    """
    tfidf_params = dict(
        max_features=10_000,
        ngram_range=(1, 2),     # unigrams + bigrams
        sublinear_tf=True,      # log-scale TF
        strip_accents="unicode",
        analyzer="word",
        token_pattern=r"\b[a-z]{2,}\b",
    )

    pipelines = {
        "Naive Bayes": Pipeline([
            ("tfidf", TfidfVectorizer(**tfidf_params)),
            ("clf",   MultinomialNB(alpha=0.1)),
        ]),
        "Logistic Regression": Pipeline([
            ("tfidf", TfidfVectorizer(**tfidf_params)),
            ("clf",   LogisticRegression(C=1.0, max_iter=1_000,
                                         solver="lbfgs", random_state=42)),
        ]),
        "Random Forest": Pipeline([
            ("tfidf", TfidfVectorizer(**tfidf_params, max_features=5_000)),
            ("clf",   RandomForestClassifier(n_estimators=200, max_depth=None,
                                              n_jobs=-1, random_state=42)),
        ]),
    }
    return pipelines


# =============================================================================
# 6. TRAIN & EVALUATE
# =============================================================================

def train_and_evaluate(pipelines: dict, X_train, X_test, y_train, y_test,
                       cv_folds: int = 5, save_plots: bool = False,
                       plot_dir: str = "plots") -> pd.DataFrame:
    """
    For each model:
        • Stratified k-fold cross-validation on training data
        • Fit on full training set
        • Evaluate on held-out test set
        • Plot confusion matrix and ROC curve

    Returns
    -------
    results_df : pd.DataFrame with per-model metrics
    """
    os.makedirs(plot_dir, exist_ok=True)
    skf     = StratifiedKFold(n_splits=cv_folds, shuffle=True, random_state=42)
    results = []

    print("\n" + "=" * 60)
    print("  MODEL TRAINING & EVALUATION")
    print("=" * 60)

    for name, pipeline in pipelines.items():
        print(f"\n── {name} ──")

        # Cross-validation
        cv_scores = cross_val_score(pipeline, X_train, y_train,
                                    cv=skf, scoring="f1", n_jobs=-1)
        print(f"   CV F1 (mean ± std): {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

        # Train
        pipeline.fit(X_train, y_train)

        # Predict
        y_pred  = pipeline.predict(X_test)
        y_proba = (pipeline.predict_proba(X_test)[:, 1]
                   if hasattr(pipeline.named_steps["clf"], "predict_proba")
                   else None)

        # Metrics
        acc  = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred, zero_division=0)
        rec  = recall_score(y_test, y_pred, zero_division=0)
        f1   = f1_score(y_test, y_pred, zero_division=0)

        print(f"   Accuracy : {acc:.4f}")
        print(f"   Precision: {prec:.4f}")
        print(f"   Recall   : {rec:.4f}")
        print(f"   F1-Score : {f1:.4f}")
        print("\n   Classification Report:")
        print(classification_report(y_test, y_pred, target_names=["ham", "spam"]))

        results.append({
            "Model": name,
            "CV F1 (mean)": round(cv_scores.mean(), 4),
            "CV F1 (std)":  round(cv_scores.std(), 4),
            "Accuracy":     round(acc, 4),
            "Precision":    round(prec, 4),
            "Recall":       round(rec, 4),
            "F1-Score":     round(f1, 4),
        })

        # ── Confusion matrix ──────────────────────────────────────────────────
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        fig.suptitle(f"{name} — Test Set Evaluation", fontsize=13, fontweight="bold")

        cm = confusion_matrix(y_test, y_pred)
        disp = ConfusionMatrixDisplay(cm, display_labels=["ham", "spam"])
        disp.plot(ax=axes[0], cmap="Blues", colorbar=False)
        axes[0].set_title("Confusion Matrix")

        # ── ROC curve ────────────────────────────────────────────────────────
        if y_proba is not None:
            fpr, tpr, _ = roc_curve(y_test, y_proba)
            roc_auc      = auc(fpr, tpr)
            axes[1].plot(fpr, tpr, color="#1565C0", lw=2,
                         label=f"AUC = {roc_auc:.4f}")
            axes[1].plot([0, 1], [0, 1], "k--", lw=1)
            axes[1].fill_between(fpr, tpr, alpha=0.1, color="#1565C0")
            axes[1].set_xlabel("False Positive Rate")
            axes[1].set_ylabel("True Positive Rate")
            axes[1].set_title("ROC Curve")
            axes[1].legend(loc="lower right")
        else:
            axes[1].set_visible(False)

        plt.tight_layout()
        fname = name.lower().replace(" ", "_")
        _save_or_show(fig, save_plots,
                      os.path.join(plot_dir, f"07_{fname}_eval.png"))

    results_df = pd.DataFrame(results).sort_values("F1-Score", ascending=False)
    return results_df


# =============================================================================
# 7. MODEL COMPARISON PLOT
# =============================================================================

def plot_model_comparison(results_df: pd.DataFrame, save_plots: bool = False,
                           plot_dir: str = "plots") -> None:
    """Bar-chart comparing all models across key metrics."""
    os.makedirs(plot_dir, exist_ok=True)

    metrics = ["Accuracy", "Precision", "Recall", "F1-Score"]
    x       = np.arange(len(metrics))
    width   = 0.22
    colors  = ["#1976D2", "#388E3C", "#F57C00"]

    fig, ax = plt.subplots(figsize=(12, 6))
    for i, (_, row) in enumerate(results_df.iterrows()):
        vals = [row[m] for m in metrics]
        ax.bar(x + i * width, vals, width, label=row["Model"],
               color=colors[i % len(colors)], edgecolor="white", alpha=0.9)
        for j, v in enumerate(vals):
            ax.text(x[j] + i * width, v + 0.003, f"{v:.3f}",
                    ha="center", va="bottom", fontsize=8)

    ax.set_xticks(x + width)
    ax.set_xticklabels(metrics, fontsize=11)
    ax.set_ylim(0.85, 1.02)
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
    ax.set_ylabel("Score")
    ax.set_title("Model Comparison — Test Set Metrics", fontsize=13, fontweight="bold")
    ax.legend(title="Model")

    plt.tight_layout()
    _save_or_show(fig, save_plots,
                  os.path.join(plot_dir, "08_model_comparison.png"))


# =============================================================================
# 8. FEATURE IMPORTANCE  (top TF-IDF tokens per class)
# =============================================================================

def plot_top_tfidf_features(pipeline, model_name: str, top_n: int = 20,
                             save_plots: bool = False,
                             plot_dir: str = "plots") -> None:
    """
    For Logistic Regression and Naive Bayes, plot the most important
    TF-IDF features for the spam class.
    """
    clf   = pipeline.named_steps["clf"]
    tfidf = pipeline.named_steps["tfidf"]
    names = tfidf.get_feature_names_out()

    if hasattr(clf, "coef_"):          # Logistic Regression
        scores = clf.coef_[0]
    elif hasattr(clf, "feature_log_prob_"):  # MultinomialNB
        scores = clf.feature_log_prob_[1] - clf.feature_log_prob_[0]
    else:
        print(f"   Feature importance not available for {model_name}.")
        return

    top_idx    = np.argsort(scores)[-top_n:]
    top_names  = names[top_idx]
    top_scores = scores[top_idx]

    fig, ax = plt.subplots(figsize=(10, 7))
    bars = ax.barh(top_names, top_scores, color="#E53935", edgecolor="none")
    ax.set_title(f"{model_name} — Top {top_n} Spam Indicators (TF-IDF)",
                 fontsize=13, fontweight="bold")
    ax.set_xlabel("Feature Weight / Log-Probability Difference")
    plt.tight_layout()
    fname = model_name.lower().replace(" ", "_")
    _save_or_show(fig, save_plots,
                  os.path.join(plot_dir, f"09_{fname}_features.png"))


# =============================================================================
# 9. PREDICT ON NEW MESSAGES
# =============================================================================

def predict_messages(pipeline, messages: list, le: LabelEncoder) -> pd.DataFrame:
    """
    Predict spam/ham for a list of raw text messages using the best pipeline.

    Parameters
    ----------
    pipeline : fitted sklearn Pipeline
    messages : list of str
    le       : fitted LabelEncoder (for decoding labels)

    Returns
    -------
    pd.DataFrame with columns: message, prediction, spam_probability
    """
    clean   = [preprocess_text(m) for m in messages]
    preds   = pipeline.predict(clean)
    labels  = le.inverse_transform(preds)
    probas  = (pipeline.predict_proba(clean)[:, 1]
               if hasattr(pipeline.named_steps["clf"], "predict_proba") else [None] * len(messages))

    result = pd.DataFrame({
        "message":          messages,
        "prediction":       labels,
        "spam_probability": [round(float(p), 4) if p is not None else None for p in probas],
    })
    return result


# =============================================================================
# 10. MAIN
# =============================================================================

def main():
    # ── Configuration ─────────────────────────────────────────────────────────
    # ▼▼▼  SET YOUR DATASET PATH HERE  ▼▼▼
    DATASET_PATH = "data/SMSSpamCollection"   # ← change to your file path
    # Examples:
    #   DATASET_PATH = "data/spam.csv"
    #   DATASET_PATH = "data/emails.csv"

    SAVE_PLOTS   = False      # True  → save PNGs;  False → display interactively
    PLOT_DIR     = "plots"    # directory where plots are saved (created automatically)
    TEST_SIZE    = 0.20       # 80/20 train-test split
    CV_FOLDS     = 5          # stratified k-fold for cross-validation
    RANDOM_STATE = 42
    # ────────────────────────────────────────────────────────────────────────────

    print("=" * 60)
    print("  SPAM DETECTION — MSc Data Science Project")
    print("=" * 60)

    # 1. Load
    df = load_dataset(DATASET_PATH)

    # 2. EDA
    run_eda(df, save_plots=SAVE_PLOTS, plot_dir=PLOT_DIR)

    # 3. Preprocess
    df = preprocess_dataframe(df)

    # 4. Split
    X_train, X_test, y_train, y_test, le = prepare_features(
        df, test_size=TEST_SIZE, random_state=RANDOM_STATE
    )

    # 5. Build pipelines
    pipelines = build_pipelines()

    # 6. Train & evaluate
    results_df = train_and_evaluate(
        pipelines, X_train, X_test, y_train, y_test,
        cv_folds=CV_FOLDS, save_plots=SAVE_PLOTS, plot_dir=PLOT_DIR
    )

    # 7. Summary table
    print("\n" + "=" * 60)
    print("  RESULTS SUMMARY  (sorted by F1-Score)")
    print("=" * 60)
    print(results_df.to_string(index=False))

    # 8. Comparison plot
    plot_model_comparison(results_df, save_plots=SAVE_PLOTS, plot_dir=PLOT_DIR)

    # 9. Best model — feature importance
    best_name = results_df.iloc[0]["Model"]
    best_pipe = pipelines[best_name]
    print(f"\n✔  Best model: {best_name}")
    plot_top_tfidf_features(best_pipe, best_name,
                             save_plots=SAVE_PLOTS, plot_dir=PLOT_DIR)

    # 10. Predict new examples
    sample_messages = [
        "Congratulations! You've won a FREE iPhone. Click here to claim now!",
        "Hey, are we still on for lunch tomorrow?",
        "URGENT: Your bank account has been suspended. Call 0800 000 0000 immediately.",
        "Can you send me the notes from today's lecture?",
        "WIN £1000 cash! Text WIN to 80888 now. T&C apply.",
    ]
    print("\n── Sample Predictions ──")
    preds_df = predict_messages(best_pipe, sample_messages, le)
    print(preds_df.to_string(index=False))

    print("\n✔  Pipeline complete. Happy publishing! 🚀")


if __name__ == "__main__":
    main()